<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0-B Initialized Fixed Grid Backtest

V0-B starts as a real spot-grid portfolio: part of the capital stays in USDT for BUY grids below the starting market price, while the remaining capital is converted to BTC at the starting market price to seed SELL targets above the market.

**Execution assumptions**
- Initial BTC inventory is created at the first candle Open.
- A grid slot is initialized with BTC when its SELL target is above the starting market price.
- BUY orders are only triggered by downward crossings for grid slots that are not currently holding BTC.
- Existing SELL targets are processed before new BUYs in each 1-minute candle.
- A new downward-crossing BUY cannot SELL in the same candle.
- A level sold in the current candle cannot rebuy in that candle.
- Same-candle SELL proceeds are not reused for BUYs.

This notebook is still a backtest/shadow-readiness model. It does not send Binance orders.


## 1. Trading Configuration

Only parameters that define how the strategy trades belong here.


In [ ]:
# Trading instrument
SYMBOL = "BTCUSDT"

# Trading capital
INITIAL_CAPITAL = 3000.0

# Fixed arithmetic grid
GRID_FLOOR = 38000.0
GRID_CEILING = 127000.0
GRID_GAP = 1000.0

# Trading fees
BUY_FEE = 0.001
SELL_FEE = 0.001


## 2. Backtest Configuration

These parameters define the historical simulation only. They do not change the grid logic.


In [ ]:
TIMEFRAME = "1m"
START_DATE = "2024-01-01"
END_DATE = "2026-01-01"


## 3. System Configuration

File locations and GitHub output settings live here.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import base64
import bisect
import heapq
import json
import os

import numpy as np
import pandas as pd
import requests

DATA_DIR = "/content/drive/MyDrive/03.Trading/00.Live Trading"

REPO = "natdanaiii/Trading"
BRANCH = "main"

GITHUB_SUMMARY_LOG_PATH = "logs/latest_v0_backtest_log.json"
GITHUB_TRADE_HISTORY_PATH = "logs/latest_v0_trade_history.csv"

LOCAL_SUMMARY_LOG_PATH = "/content/latest_v0_backtest_log.json"
LOCAL_TRADE_HISTORY_PATH = "/content/latest_v0_trade_history.csv"


## 4. Derived Parameters

These values are calculated from Trading Configuration and should not be edited manually.


In [ ]:
def derive_grid_parameters(capital, floor, ceiling, gap):
    """Validate the fixed grid and derive grid count and order size."""

    if capital <= 0:
        raise ValueError("INITIAL_CAPITAL must be greater than 0.")
    if floor <= 0:
        raise ValueError("GRID_FLOOR must be greater than 0.")
    if ceiling <= floor:
        raise ValueError("GRID_CEILING must be greater than GRID_FLOOR.")
    if gap <= 0:
        raise ValueError("GRID_GAP must be greater than 0.")

    raw_grid_count = (ceiling - floor) / gap

    if not np.isclose(raw_grid_count, round(raw_grid_count)):
        raise ValueError(
            "(GRID_CEILING - GRID_FLOOR) must be exactly divisible by GRID_GAP."
        )

    number_of_grids = int(round(raw_grid_count))
    capital_per_grid = capital / number_of_grids

    return number_of_grids, capital_per_grid


NUMBER_OF_GRIDS, CAPITAL_PER_GRID = derive_grid_parameters(
    INITIAL_CAPITAL,
    GRID_FLOOR,
    GRID_CEILING,
    GRID_GAP,
)

print(f"Number of Grids : {NUMBER_OF_GRIDS}")
print(f"Capital / Grid  : {CAPITAL_PER_GRID:,.6f} USDT")


## 5. Load Market Data


In [ ]:
def load_market_data(symbol, timeframe, data_dir, start_date, end_date):
    """Load, validate, sort, and filter historical OHLCV data."""

    file_path = os.path.join(
        data_dir,
        f"{symbol}-{timeframe}-combined.csv",
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    data = pd.read_csv(file_path)

    required_columns = {
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
    }
    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise ValueError(f"Missing columns: {sorted(missing_columns)}")

    data["open_time"] = pd.to_datetime(data["open_time"], utc=True)

    numeric_columns = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]
    data[numeric_columns] = data[numeric_columns].astype(float)

    start_time = pd.Timestamp(start_date, tz="UTC")
    end_time = pd.Timestamp(end_date, tz="UTC")

    data = (
        data
        .drop_duplicates("open_time")
        .sort_values("open_time")
        .loc[
            lambda df:
            (df["open_time"] >= start_time)
            & (df["open_time"] < end_time)
        ]
        .reset_index(drop=True)
    )

    if data.empty:
        raise ValueError("No market data inside the selected period.")

    return data


## 6. Build Fixed Grid


In [ ]:
def build_fixed_grid_table(
    floor,
    ceiling,
    gap,
    capital_per_grid,
):
    """Build the fixed arithmetic grid.

    Each row is one grid slot:
    BUY at buy_price -> SELL at sell_target.
    """

    buy_prices = np.arange(
        floor,
        ceiling,
        gap,
        dtype=float,
    )
    sell_targets = buy_prices + gap

    return pd.DataFrame(
        {
            "grid_id": np.arange(1, len(buy_prices) + 1),
            "buy_price": buy_prices,
            "sell_target": sell_targets,
            "order_size_usdt": float(capital_per_grid),
        }
    )


## 7. Position Helpers

The same position structure is used for both:
- initial BTC inventory created at the starting market price, and
- later grid BUYs triggered by downward crossings.


In [ ]:
def create_position(
    trade_id,
    grid_index,
    buy_time,
    buy_price,
    sell_target,
    order_size_usdt,
    portfolio_value_at_buy,
    buy_fee,
    sell_fee,
):
    """Create one spot position and pre-calculate its SELL economics."""

    gross_btc = order_size_usdt / buy_price
    buy_fee_btc = gross_btc * buy_fee
    btc_amount = gross_btc - buy_fee_btc

    gross_sell_usdt = btc_amount * sell_target
    sell_fee_usdt = gross_sell_usdt * sell_fee
    net_sell_usdt = gross_sell_usdt - sell_fee_usdt

    realized_pnl_if_sold = net_sell_usdt - order_size_usdt

    return {
        "trade_id": int(trade_id),
        "grid_index": int(grid_index),
        "status": "OPEN",
        "buy_time": buy_time,
        "buy_price": float(buy_price),
        "sell_target": float(sell_target),
        "sell_time": pd.NaT,
        "order_size_usdt": float(order_size_usdt),
        "portfolio_value_at_buy": float(portfolio_value_at_buy),
        "order_pct_of_portfolio": float(
            order_size_usdt / portfolio_value_at_buy
        ),
        "portfolio_value_at_sell": np.nan,
        "btc_amount": float(btc_amount),
        "buy_fee_btc": float(buy_fee_btc),
        "sell_fee_usdt": float(sell_fee_usdt),
        "net_sell_usdt": float(net_sell_usdt),
        "realized_pnl_if_sold": float(realized_pnl_if_sold),
        "net_pnl": np.nan,
    }


def initialize_portfolio(
    data,
    grid,
    initial_capital,
    buy_fee,
    sell_fee,
):
    """Create the V0-B starting USDT/BTC allocation at the first candle Open."""

    start_time = data.iloc[0]["open_time"]
    start_price = float(data.iloc[0]["open"])

    grid_floor = float(grid["buy_price"].min())
    grid_ceiling = float(grid["sell_target"].max())

    if not (grid_floor < start_price < grid_ceiling):
        raise ValueError(
            "Starting market price must be strictly inside the configured grid range."
        )

    positions = {}
    active_trade_by_grid = {}
    sell_heap = []
    trade_events = []

    cash = float(initial_capital)
    btc = 0.0
    initial_buy_fee_usdt = 0.0
    trade_id = 0
    event_id = 0

    # Every grid slot whose SELL target is above market starts with BTC.
    initial_grid_indices = grid.index[
        grid["sell_target"] > start_price
    ].tolist()

    for grid_index in initial_grid_indices:
        row = grid.loc[grid_index]
        order_size = float(row["order_size_usdt"])

        trade_id += 1

        position = create_position(
            trade_id=trade_id,
            grid_index=grid_index,
            buy_time=start_time,
            buy_price=start_price,
            sell_target=float(row["sell_target"]),
            order_size_usdt=order_size,
            portfolio_value_at_buy=initial_capital,
            buy_fee=buy_fee,
            sell_fee=sell_fee,
        )

        cash_before = cash
        btc_before = btc

        cash -= order_size
        btc += position["btc_amount"]

        buy_fee_usdt = position["buy_fee_btc"] * start_price
        initial_buy_fee_usdt += buy_fee_usdt

        positions[trade_id] = position
        active_trade_by_grid[grid_index] = trade_id

        heapq.heappush(
            sell_heap,
            (position["sell_target"], trade_id),
        )

        event_id += 1
        trade_events.append(
            {
                "event_id": event_id,
                "time": start_time,
                "side": "BUY",
                "trade_id": trade_id,
                "grid_index": grid_index,
                "price": start_price,
                "cash_movement": -order_size,
                "grid_cashflow": 0.0,
                "cash_before": cash_before,
                "cash_after": cash,
                "btc_before": btc_before,
                "btc_after": btc,
                "portfolio_value_before": initial_capital,
                "initialization_trade": True,
            }
        )

    initialization = {
        "start_time": start_time,
        "start_price": start_price,
        "initial_sell_positions": len(initial_grid_indices),
        "initial_buy_levels": len(grid) - len(initial_grid_indices),
        "initial_cash": float(cash),
        "initial_btc": float(btc),
        "initial_btc_cost_usdt": float(
            len(initial_grid_indices)
            * grid["order_size_usdt"].iloc[0]
        ),
        "initial_buy_fee_usdt": float(initial_buy_fee_usdt),
    }

    return {
        "cash": cash,
        "btc": btc,
        "positions": positions,
        "active_trade_by_grid": active_trade_by_grid,
        "sell_heap": sell_heap,
        "trade_events": trade_events,
        "next_trade_id": trade_id,
        "next_event_id": event_id,
        "initialization": initialization,
    }


## 8. Performance Statistics


In [ ]:
def performance_stats(data, equity, initial_capital):
    """Calculate return, drawdown, annualized return, and Calmar ratio."""

    running_peak = np.maximum.accumulate(equity)
    drawdown = equity / running_peak - 1.0

    final_equity = float(equity[-1])
    max_drawdown = float(drawdown.min())
    net_return = final_equity / initial_capital - 1.0

    elapsed_days = (
        data["open_time"].iloc[-1]
        - data["open_time"].iloc[0]
    ).total_seconds() / 86400.0

    annualized_return = np.nan

    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = (
            np.log(final_equity / initial_capital)
            * (365.25 / elapsed_days)
        )

        if annualized_log_growth < 700:
            annualized_return = float(
                np.expm1(annualized_log_growth)
            )

    calmar_ratio = np.nan

    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar_ratio = float(
            annualized_return / abs(max_drawdown)
        )

    return {
        "final_equity": final_equity,
        "net_return": net_return,
        "annualized_return": annualized_return,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar_ratio,
        "drawdown": drawdown,
    }


## 9. Trade History Helper

One user-facing table contains every position ever opened. CLOSED and OPEN trades stay in the same table.


In [ ]:
def build_trade_history(positions, final_time):
    """Build the single user-facing historical trade table."""

    rows = []

    for trade_id in sorted(positions):
        position = positions[trade_id]

        if position["status"] == "CLOSED":
            holding_time = (
                position["sell_time"]
                - position["buy_time"]
            )
        else:
            holding_time = (
                final_time
                - position["buy_time"]
            )

        rows.append(
            {
                "Trade ID": position["trade_id"],
                "Status": position["status"],
                "Buy Time": position["buy_time"],
                "Buy Price": position["buy_price"],
                "Sell Target": position["sell_target"],
                "Sell Time": position["sell_time"],
                "Order Size (USDT)": position["order_size_usdt"],
                "Portfolio Value at Buy": position["portfolio_value_at_buy"],
                "Order % of Portfolio": (
                    position["order_pct_of_portfolio"] * 100.0
                ),
                "Portfolio Value at Sell": position["portfolio_value_at_sell"],
                "Net P&L": position["net_pnl"],
                "Holding Time": str(holding_time),
            }
        )

    history = pd.DataFrame(rows)

    if not history.empty:
        history = history.sort_values(
            ["Buy Time", "Trade ID"]
        ).reset_index(drop=True)

    return history


## 10. V0-B Backtest Engine

Initialization occurs once at the first candle Open.

Then for every 1-minute candle:

1. Process existing SELL targets.
2. Process downward BUY crossings for available grid slots.
3. Mark portfolio equity at the candle Close.


In [ ]:
def run_initialized_grid_backtest(
    df_price,
    grid,
    initial_capital,
    buy_fee,
    sell_fee,
):
    """Run the V0-B initialized fixed-grid strategy."""

    data = (
        df_price
        .sort_values("open_time")
        .reset_index(drop=True)
    )
    grid = (
        grid
        .sort_values("buy_price")
        .reset_index(drop=True)
        .copy()
    )

    if data.empty:
        raise ValueError("df_price is empty.")
    if grid.empty:
        raise ValueError("grid is empty.")

    buy_prices = grid["buy_price"].to_numpy(float)
    buy_price_list = buy_prices.tolist()

    state = initialize_portfolio(
        data=data,
        grid=grid,
        initial_capital=initial_capital,
        buy_fee=buy_fee,
        sell_fee=sell_fee,
    )

    cash = state["cash"]
    btc = state["btc"]
    positions = state["positions"]
    active_trade_by_grid = state["active_trade_by_grid"]
    sell_heap = state["sell_heap"]
    trade_events = state["trade_events"]
    initialization = state["initialization"]

    trade_id = state["next_trade_id"]
    event_id = state["next_event_id"]

    realized_profit = 0.0
    total_buy_fee_usdt = initialization["initial_buy_fee_usdt"]
    total_sell_fee_usdt = 0.0
    completed_cycles = 0

    row_count = len(data)
    equity_values = np.empty(row_count)
    cash_values = np.empty(row_count)
    btc_values = np.empty(row_count)

    previous_close = None

    for row_index, candle in enumerate(data.itertuples(index=False)):
        timestamp = candle.open_time
        open_price = float(candle.open)
        high_price = float(candle.high)
        low_price = float(candle.low)
        close_price = float(candle.close)

        # BUYs in this candle may only use cash available at candle start.
        cash_at_candle_start = cash
        sold_this_candle = set()

        # ----------------------------------------------------
        # 1) Process existing SELL targets
        # ----------------------------------------------------
        while sell_heap and sell_heap[0][0] <= high_price:
            _, closing_trade_id = heapq.heappop(sell_heap)
            position = positions[closing_trade_id]

            if position["status"] != "OPEN":
                continue

            grid_index = position["grid_index"]

            cash_before = cash
            btc_before = btc

            portfolio_value_at_sell = (
                cash_before
                + btc_before * position["sell_target"]
            )

            cash += position["net_sell_usdt"]
            btc -= position["btc_amount"]

            if abs(btc) < 1e-12:
                btc = 0.0

            position["status"] = "CLOSED"
            position["sell_time"] = timestamp
            position["portfolio_value_at_sell"] = (
                portfolio_value_at_sell
            )
            position["net_pnl"] = position["realized_pnl_if_sold"]

            realized_profit += position["net_pnl"]
            total_sell_fee_usdt += position["sell_fee_usdt"]
            completed_cycles += 1

            active_trade_by_grid.pop(grid_index, None)
            sold_this_candle.add(grid_index)

            event_id += 1
            trade_events.append(
                {
                    "event_id": event_id,
                    "time": timestamp,
                    "side": "SELL",
                    "trade_id": closing_trade_id,
                    "grid_index": grid_index,
                    "price": position["sell_target"],
                    "cash_movement": position["net_sell_usdt"],
                    "grid_cashflow": position["net_pnl"],
                    "cash_before": cash_before,
                    "cash_after": cash,
                    "btc_before": btc_before,
                    "btc_after": btc,
                    "portfolio_value_before": portfolio_value_at_sell,
                    "initialization_trade": False,
                }
            )

        # ----------------------------------------------------
        # 2) Process downward BUY crossings
        # ----------------------------------------------------
        buy_budget = cash_at_candle_start

        downward_start = (
            open_price
            if previous_close is None
            else max(previous_close, open_price)
        )

        if low_price < downward_start:
            first_index = bisect.bisect_left(
                buy_price_list,
                low_price,
            )
            stop_index = bisect.bisect_left(
                buy_price_list,
                downward_start,
            )

            # Higher crossed levels are reached first as price falls.
            for grid_index in range(
                stop_index - 1,
                first_index - 1,
                -1,
            ):
                if grid_index in active_trade_by_grid:
                    continue
                if grid_index in sold_this_candle:
                    continue

                row = grid.loc[grid_index]
                order_size = float(row["order_size_usdt"])

                if buy_budget + 1e-12 < order_size:
                    break

                buy_price = float(row["buy_price"])
                sell_target = float(row["sell_target"])

                cash_before = cash
                btc_before = btc

                portfolio_value_at_buy = (
                    cash_before
                    + btc_before * buy_price
                )

                trade_id += 1
                position = create_position(
                    trade_id=trade_id,
                    grid_index=grid_index,
                    buy_time=timestamp,
                    buy_price=buy_price,
                    sell_target=sell_target,
                    order_size_usdt=order_size,
                    portfolio_value_at_buy=portfolio_value_at_buy,
                    buy_fee=buy_fee,
                    sell_fee=sell_fee,
                )

                buy_budget -= order_size
                cash -= order_size
                btc += position["btc_amount"]

                total_buy_fee_usdt += (
                    position["buy_fee_btc"]
                    * buy_price
                )

                positions[trade_id] = position
                active_trade_by_grid[grid_index] = trade_id

                heapq.heappush(
                    sell_heap,
                    (sell_target, trade_id),
                )

                event_id += 1
                trade_events.append(
                    {
                        "event_id": event_id,
                        "time": timestamp,
                        "side": "BUY",
                        "trade_id": trade_id,
                        "grid_index": grid_index,
                        "price": buy_price,
                        "cash_movement": -order_size,
                        "grid_cashflow": 0.0,
                        "cash_before": cash_before,
                        "cash_after": cash,
                        "btc_before": btc_before,
                        "btc_after": btc,
                        "portfolio_value_before": portfolio_value_at_buy,
                        "initialization_trade": False,
                    }
                )

        # ----------------------------------------------------
        # 3) Mark portfolio at candle Close
        # ----------------------------------------------------
        equity_values[row_index] = cash + btc * close_price
        cash_values[row_index] = cash
        btc_values[row_index] = btc

        previous_close = close_price

    stats = performance_stats(
        data=data,
        equity=equity_values,
        initial_capital=initial_capital,
    )

    final_close = float(data["close"].iloc[-1])
    final_time = data["open_time"].iloc[-1]

    # Mark remaining OPEN positions at the final Close.
    for position in positions.values():
        if position["status"] == "OPEN":
            position["net_pnl"] = (
                position["btc_amount"] * final_close
                - position["order_size_usdt"]
            )

    equity_curve = pd.DataFrame(
        {
            "open_time": data["open_time"],
            "close": data["close"],
            "cash": cash_values,
            "btc": btc_values,
            "equity": equity_values,
            "drawdown": stats["drawdown"],
        }
    )

    trade_log = pd.DataFrame(trade_events)

    trade_history = build_trade_history(
        positions=positions,
        final_time=final_time,
    )

    summary = {
        "initial_capital": float(initial_capital),
        "initial_market_price": initialization["start_price"],
        "initial_cash": initialization["initial_cash"],
        "initial_btc": initialization["initial_btc"],
        "initial_sell_positions": initialization["initial_sell_positions"],
        "initial_buy_levels": initialization["initial_buy_levels"],
        "final_equity": stats["final_equity"],
        "net_return": stats["net_return"],
        "annualized_return": stats["annualized_return"],
        "max_drawdown": stats["max_drawdown"],
        "calmar_ratio": stats["calmar_ratio"],
        "completed_cycles": int(completed_cycles),
        "open_positions": int(
            sum(
                position["status"] == "OPEN"
                for position in positions.values()
            )
        ),
        "final_cash": float(cash),
        "final_btc": float(btc),
        "realized_profit": float(realized_profit),
        "unrealized_pnl": float(
            stats["final_equity"]
            - initial_capital
            - realized_profit
        ),
        "total_fee_usdt_equiv": float(
            total_buy_fee_usdt
            + total_sell_fee_usdt
        ),
    }

    return {
        "summary": summary,
        "initialization": initialization,
        "trade_log": trade_log,
        "trade_history": trade_history,
        "equity_curve": equity_curve,
        "positions": positions,
    }


## 11. System Audit

The audit checks accounting identities and V0-B initialization consistency.


In [ ]:
def audit_v0b(result, initial_capital, grid):
    """Run accounting and initialization consistency checks."""

    summary = result["summary"]
    initialization = result["initialization"]
    trade_log = result["trade_log"]
    equity = result["equity_curve"]
    trade_history = result["trade_history"]

    cash_movement = (
        trade_log["cash_movement"].sum()
        if len(trade_log)
        else 0.0
    )
    grid_cashflow = (
        trade_log["grid_cashflow"].sum()
        if len(trade_log)
        else 0.0
    )

    initial_equity_plus_fee = (
        initialization["initial_cash"]
        + initialization["initial_btc"]
        * initialization["start_price"]
        + initialization["initial_buy_fee_usdt"]
    )

    closed_trades = int(
        (trade_history["Status"] == "CLOSED").sum()
    )

    return {
        "cash_reconciliation": (
            abs(
                initial_capital
                + cash_movement
                - summary["final_cash"]
            )
            <= 1e-8
        ),
        "realized_profit_reconciliation": (
            abs(
                grid_cashflow
                - summary["realized_profit"]
            )
            <= 1e-8
        ),
        "equity_identity": (
            float(
                np.max(
                    np.abs(
                        equity["cash"]
                        + equity["btc"] * equity["close"]
                        - equity["equity"]
                    )
                )
            )
            <= 1e-8
        ),
        "cash_never_negative": (
            float(equity["cash"].min()) >= -1e-8
        ),
        "initial_allocation_reconciliation": (
            abs(
                initial_equity_plus_fee
                - initial_capital
            )
            <= 1e-8
        ),
        "all_grid_slots_initialized_or_reserved": (
            initialization["initial_sell_positions"]
            + initialization["initial_buy_levels"]
            == len(grid)
        ),
        "closed_trade_count_reconciliation": (
            closed_trades
            == summary["completed_cycles"]
        ),
    }


## 12. Run Backtest & Review Results


In [ ]:
df_1m = load_market_data(
    symbol=SYMBOL,
    timeframe=TIMEFRAME,
    data_dir=DATA_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
)

grid = build_fixed_grid_table(
    floor=GRID_FLOOR,
    ceiling=GRID_CEILING,
    gap=GRID_GAP,
    capital_per_grid=CAPITAL_PER_GRID,
)

result = run_initialized_grid_backtest(
    df_price=df_1m,
    grid=grid,
    initial_capital=INITIAL_CAPITAL,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE,
)

summary = result["summary"]
initialization = result["initialization"]
trade_history = result["trade_history"]

audit_checks = audit_v0b(
    result=result,
    initial_capital=INITIAL_CAPITAL,
    grid=grid,
)

AUDIT_STATUS = (
    "PASS"
    if all(audit_checks.values())
    else "FAIL"
)

initial_btc_allocation_pct = (
    initialization["initial_btc_cost_usdt"]
    / INITIAL_CAPITAL
    * 100.0
)

print("===== V0-B TRADING CONFIGURATION =====")
print(f"Symbol                 : {SYMBOL}")
print(f"Initial Capital        : {INITIAL_CAPITAL:,.2f} USDT")
print(f"Floor                  : {GRID_FLOOR:,.2f} USDT")
print(f"Ceiling                : {GRID_CEILING:,.2f} USDT")
print(f"Gap                    : {GRID_GAP:,.2f} USDT")
print(f"Number of Grids        : {NUMBER_OF_GRIDS}")
print(f"Order Size / Grid      : {CAPITAL_PER_GRID:,.6f} USDT")

print("\n===== INITIAL GRID ALLOCATION =====")
print(f"Starting Market Price  : {initialization['start_price']:,.2f} USDT")
print(f"Initial SELL Positions : {initialization['initial_sell_positions']}")
print(f"Initial BUY Levels     : {initialization['initial_buy_levels']}")
print(f"Initial Cash           : {initialization['initial_cash']:,.2f} USDT")
print(f"Initial BTC            : {initialization['initial_btc']:.8f} BTC")
print(f"BTC Capital Allocation : {initial_btc_allocation_pct:.2f}%")

if initial_btc_allocation_pct > 80:
    print(
        "WARNING: More than 80% of initial capital is allocated to BTC. "
        "Review Floor/Ceiling relative to the starting market price before live use."
    )

print("\n===== V0-B RESULT =====")
result_fields = [
    "final_equity",
    "net_return",
    "annualized_return",
    "max_drawdown",
    "calmar_ratio",
    "completed_cycles",
    "open_positions",
    "final_cash",
    "final_btc",
    "realized_profit",
    "unrealized_pnl",
    "total_fee_usdt_equiv",
]

for field in result_fields:
    print(f"{field:32s}: {summary[field]}")

print("\n===== V0-B AUDIT =====")
for name, passed in audit_checks.items():
    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print(f"{'Overall':40s}: {AUDIT_STATUS}")

if AUDIT_STATUS != "PASS":
    raise AssertionError("V0-B AUDIT FAILED")


## 13. Trade History

This is the single user-facing historical trade table.

- `CLOSED`: Net P&L is realized after BUY and SELL fees.
- `OPEN`: Net P&L is unrealized at the final backtest Close; no future SELL fee is assumed yet.
- Initial SELL-side inventory appears as BUYs at the first candle Open because BTC must actually be acquired to seed those SELL orders.


In [ ]:
pd.set_option("display.max_columns", None)

display(
    trade_history[
        [
            "Trade ID",
            "Status",
            "Buy Time",
            "Buy Price",
            "Sell Target",
            "Sell Time",
            "Order Size (USDT)",
            "Portfolio Value at Buy",
            "Order % of Portfolio",
            "Portfolio Value at Sell",
            "Net P&L",
            "Holding Time",
        ]
    ]
)


## 14. Export Summary Log + Trade History


In [ ]:
def json_safe(value):
    """Convert NumPy/Pandas values into JSON-safe Python values."""

    if value is pd.NaT or value is pd.NA:
        return None
    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return [
            json_safe(item)
            for item in value.tolist()
        ]
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        if not np.isfinite(value):
            return None
        return float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    return value


def build_summary_log():
    """Build the compact machine-readable V0-B summary log."""

    return json_safe(
        {
            "log_schema_version": 3,
            "strategy": "V0-B Initialized Fixed Grid",
            "run_info": {
                "generated_at_utc": (
                    pd.Timestamp.now(tz="UTC").isoformat()
                ),
                "repository": REPO,
                "branch": BRANCH,
                "notebook": "Grid_trading_V0.ipynb",
                "symbol": SYMBOL,
                "timeframe": TIMEFRAME,
                "start_date": START_DATE,
                "end_date": END_DATE,
                "data_rows": int(len(df_1m)),
                "data_first_time": (
                    df_1m["open_time"].min().isoformat()
                ),
                "data_last_time": (
                    df_1m["open_time"].max().isoformat()
                ),
            },
            "trading_config": {
                "initial_capital": INITIAL_CAPITAL,
                "floor": GRID_FLOOR,
                "ceiling": GRID_CEILING,
                "gap": GRID_GAP,
                "buy_fee": BUY_FEE,
                "sell_fee": SELL_FEE,
            },
            "backtest_config": {
                "timeframe": TIMEFRAME,
                "start_date": START_DATE,
                "end_date": END_DATE,
            },
            "derived": {
                "number_of_grids": NUMBER_OF_GRIDS,
                "capital_per_grid": CAPITAL_PER_GRID,
            },
            "initialization": {
                **initialization,
                "initial_btc_allocation_pct": (
                    initial_btc_allocation_pct
                ),
            },
            "market_range_diagnostics": {
                "historical_low": float(df_1m["low"].min()),
                "historical_high": float(df_1m["high"].max()),
                "candles_low_below_floor": int(
                    (df_1m["low"] < GRID_FLOOR).sum()
                ),
                "candles_high_above_ceiling": int(
                    (df_1m["high"] > GRID_CEILING).sum()
                ),
            },
            "summary": summary,
            "audit": {
                "status": AUDIT_STATUS,
                "checks": audit_checks,
            },
            "trade_history_file": (
                GITHUB_TRADE_HISTORY_PATH
            ),
        }
    )


def upload_text_to_github(
    path,
    text_content,
    commit_message,
    github_token,
):
    """Create or replace one UTF-8 text file in the configured repository."""

    api_url = (
        f"https://api.github.com/repos/{REPO}/contents/{path}"
    )

    headers = {
        "Authorization": f"Bearer {github_token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    existing = requests.get(
        api_url,
        headers=headers,
        timeout=30,
    )

    body = {
        "message": commit_message,
        "content": base64.b64encode(
            text_content.encode("utf-8")
        ).decode("utf-8"),
        "branch": BRANCH,
    }

    if existing.status_code == 200:
        body["sha"] = existing.json()["sha"]

    upload = requests.put(
        api_url,
        headers=headers,
        json=body,
        timeout=30,
    )
    upload.raise_for_status()

    return upload.json()["commit"]["sha"]


summary_log = build_summary_log()

summary_json = json.dumps(
    summary_log,
    indent=2,
    allow_nan=False,
)

trade_history_csv = trade_history.to_csv(
    index=False,
)

with open(
    LOCAL_SUMMARY_LOG_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(summary_json)

with open(
    LOCAL_TRADE_HISTORY_PATH,
    "w",
    encoding="utf-8",
    newline="",
) as file:
    file.write(trade_history_csv)

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None

if not github_token:
    print(
        "GitHub upload SKIPPED: "
        "Colab Secret 'GITHUB_TOKEN' was not found."
    )
else:
    summary_commit = upload_text_to_github(
        path=GITHUB_SUMMARY_LOG_PATH,
        text_content=summary_json,
        commit_message="Update latest V0-B backtest log",
        github_token=github_token,
    )

    trade_commit = upload_text_to_github(
        path=GITHUB_TRADE_HISTORY_PATH,
        text_content=trade_history_csv,
        commit_message="Update latest V0-B trade history",
        github_token=github_token,
    )

    print("GitHub upload: SUCCESS")
    print(
        f"Summary Log  : {GITHUB_SUMMARY_LOG_PATH}"
    )
    print(
        f"Trade History: {GITHUB_TRADE_HISTORY_PATH}"
    )
    print(f"Summary Commit: {summary_commit}")
    print(f"Trade Commit  : {trade_commit}")
